# Fine-tune PhoBERT cho Voice Control
Upload file `dataset.csv` lên Colab trước khi chạy.

In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# Load dataset
df = pd.read_csv('dataset.csv')
print(df['intent'].value_counts())
print(f'Tổng: {len(df)} mẫu')

In [ ]:
# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['intent'])
print('Classes:', list(le.classes_))
NUM_LABELS = len(le.classes_)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Load PhoBERT tokenizer
MODEL_NAME = 'vinai/phobert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = IntentDataset(X_train, y_train)
test_dataset  = IntentDataset(X_test,  y_test)

In [ ]:
# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=8)

In [ ]:
# Train
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}')

In [ ]:
# Evaluate
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].cpu().tolist())

print(classification_report(all_labels, all_preds, target_names=le.classes_))

In [ ]:
# Export model
import json, os
os.makedirs('phobert_intent', exist_ok=True)
model.save_pretrained('phobert_intent')
tokenizer.save_pretrained('phobert_intent')

# Lưu label mapping
with open('phobert_intent/label_map.json', 'w') as f:
    json.dump({i: c for i, c in enumerate(le.classes_)}, f, ensure_ascii=False)

print('Đã lưu model vào thư mục phobert_intent/')

# Zip để download
!zip -r phobert_intent.zip phobert_intent/
from google.colab import files
files.download('phobert_intent.zip')

In [ ]:
# Test nhanh
def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    idx = torch.argmax(logits).item()
    return le.classes_[idx]

tests = ['trời nóng quá', 'thôi ngủ rồi', 'mạnh lên đi', 'số hai thôi', 'tăng thêm một chút']
for t in tests:
    print(f'{t!r:30} → {predict(t)}')